In [15]:
import csv

In [16]:
import os

vcffile = '../../../hybrid_diversity/5_snp_analyses/3_aims/3_vcf/hybrid_variants_AIMs.vcf'
outdir  = '../../../hybrid_diversity/5_snp_analyses/3_aims/3_vcf/allelic_depth_per_sample/'
os.makedirs(outdir, exist_ok=True)

In [17]:
# Two reference strains that define A1 and A2
A1_STRAIN = "CC2192B4"  # homozygous for A1 alleles
A2_STRAIN = "RCC3703"   # homozygous for A2 alleles

skipped_chromosomes = ["CM083886.1", "CM083887.1", "CM083888.1", "CM083889.1", "CM083885.1"]

In [18]:
chromdict = {'CM083856.1': 'scaf1',
'CM083857.1': 'scaf2',
'CM083858.1': 'scaf3',
'CM083859.1': 'scaf4',
'CM083860.1': 'scaf5',
'CM083861.1': 'scaf6',
'CM083862.1': 'scaf7',
'CM083863.1': 'scaf8',
'CM083864.1': 'scaf9',
'CM083865.1': 'scaf10',
'CM083866.1': 'scaf11',
'CM083867.1': 'scaf12',
'CM083868.1': 'scaf13',
'CM083869.1': 'scaf14',
'CM083870.1': 'scaf15',
'CM083871.1': 'scaf16',
'CM083872.1': 'scaf17',
'CM083873.1': 'scaf18',
'CM083874.1': 'scaf19',
'CM083875.1': 'scaf20',
'CM083876.1': 'scaf21',
'CM083877.1': 'scaf22',
'CM083878.1': 'scaf23',
'CM083879.1': 'scaf24',
'CM083880.1': 'scaf25',
'CM083881.1': 'scaf26',
'CM083882.1': 'scaf27',
'CM083883.1': 'scaf28',
'CM083884.1': 'scaf29',
'CM083885.1': 'scaf30',
'CM083886.1': 'scaf31',
'CM083887.1': 'scaf32',
'CM083888.1': 'scaf33',
'CM083889.1': 'scaf34'}

In [19]:
fieldnames = [
    "CHROM", "POS", "SNP_ID", "REF", "ALT",
    "A1_allele", "A2_allele", "Sample", "A1_depth", "A2_depth", "A1_frac", "A2_frac"
]

samples = []
skipped_snps = []
n_rows = 0

In [20]:
# Open input VCF; per-sample output files are opened after parsing the header
# All sample file handles are kept open for the duration of the VCF scan,
# then closed together — this avoids holding any SNP data in memory.

writers  = {}   # sample_name -> csv.DictWriter
handles  = {}   # sample_name -> file handle

try:
    with open(vcffile) as f_in:
        for line in f_in:
            if line.startswith("##"):
                continue

            fields = line.rstrip("\n").split("\t")

            # ── Header line: open one output file per sample ──────────────────
            if line.startswith("#CHROM"):
                samples = fields[9:]
                if A1_STRAIN not in samples:
                    raise ValueError(f"Reference strain '{A1_STRAIN}' not found in VCF header.")
                if A2_STRAIN not in samples:
                    raise ValueError(f"Reference strain '{A2_STRAIN}' not found in VCF header.")
                a1_idx = samples.index(A1_STRAIN)
                a2_idx = samples.index(A2_STRAIN)

                for sample in samples:
                    safe_name = sample.replace("/", "_").replace(" ", "_")
                    path = outdir + f"{safe_name}.allelic_depth.tsv"
                    fh = open(path, "w", newline="")
                    w  = csv.DictWriter(fh, fieldnames=fieldnames, delimiter="\t")
                    w.writeheader()
                    handles[sample]  = fh
                    writers[sample]  = w
                continue

            # ── Data line ─────────────────────────────────────────────────────
            chrom, pos, snp_id, ref, alt = fields[0], fields[1], fields[2], fields[3], fields[4]
            fmt = fields[8].split(":")

            if chrom in skipped_chromosomes:
                continue

            if "AD" not in fmt or "GT" not in fmt:
                continue  # skip sites without required fields

            ad_idx = fmt.index("AD")
            gt_idx = fmt.index("GT")

            def get_gt(sample_fields):
                gf = sample_fields.split(":")
                return gf[gt_idx] if gt_idx < len(gf) else "."

            a1_gt = get_gt(fields[9 + a1_idx])
            a2_gt = get_gt(fields[9 + a2_idx])

            # Determine allele orientation from the two reference strains
            a1_alleles = set(a1_gt.replace("|", "/").split("/"))
            a2_alleles = set(a2_gt.replace("|", "/").split("/"))

            if a1_alleles == {"0"} and a2_alleles == {"1"}:
                a1_ad_pos, a2_ad_pos = 0, 1   # standard: REF=A1, ALT=A2
            elif a1_alleles == {"1"} and a2_alleles == {"0"}:
                a1_ad_pos, a2_ad_pos = 1, 0   # flipped:  ALT=A1, REF=A2
            else:
                skipped_snps.append((chrom, pos, snp_id, a1_gt, a2_gt))
                continue

            a1_base = ref if a1_ad_pos == 0 else alt
            a2_base = alt if a2_ad_pos == 1 else ref

            # Write one row to each sample's file
            for sample, geno in zip(samples, fields[9:]):
                geno_fields = geno.split(":")
                ad = geno_fields[ad_idx] if ad_idx < len(geno_fields) else "."

                if ad == "." or "," not in ad:
                    a1_dp, a2_dp, a1_frac, a2_frac = ".", ".", ".", "."
                else:
                    ad_parts = ad.split(",")
                    a1_dp = ad_parts[a1_ad_pos] if a1_ad_pos < len(ad_parts) else "."
                    a2_dp = ad_parts[a2_ad_pos] if a2_ad_pos < len(ad_parts) else "."
                    try:
                        total = int(a1_dp) + int(a2_dp)
                        a1_frac = round(int(a1_dp) / total, 4) if total > 0 else "."
                        a2_frac = round(int(a2_dp) / total, 4) if total > 0 else "."
                    except (ValueError, TypeError):
                        a1_frac, a2_frac = ".", "."

                writers[sample].writerow({
                    "CHROM":     chromdict[chrom],
                    "POS":       pos,
                    "SNP_ID":    snp_id,
                    "REF":       ref,
                    "ALT":       alt,
                    "A1_allele": a1_base,
                    "A2_allele": a2_base,
                    "Sample":    sample,
                    "A1_depth":  a1_dp,
                    "A2_depth":  a2_dp,
                    "A1_frac":   a1_frac,
                    "A2_frac":   a2_frac,
                })
                n_rows += 1

finally:
    for fh in handles.values():
        fh.close()



In [21]:
print(f"Done. {n_rows} rows written across {len(handles)} files in {outdir}")
print(f"SNPs skipped (reference strains not cleanly homozygous): {len(skipped_snps)}")
if skipped_snps:
    print("Skipped SNPs:")
    for s in skipped_snps:
        print(f"  {s[0]}:{s[1]} ({s[2]})  {A1_STRAIN}={s[3]}  {A2_STRAIN}={s[4]}")


Done. 8729688 rows written across 33 files in ../../../hybrid_diversity/5_snp_analyses/3_aims/3_vcf/allelic_depth_per_sample/
SNPs skipped (reference strains not cleanly homozygous): 0


In [22]:
! head ../../../hybrid_diversity/5_snp_analyses/3_aims/3_vcf/allelic_depth_per_sample/12A1-SC1.allelic_depth.tsv

CHROM	POS	SNP_ID	REF	ALT	A1_allele	A2_allele	Sample	A1_depth	A2_depth	A1_frac	A2_frac
scaf1	3067	CHROM:3067:G:A	G	A	G	A	12A1-SC1	26	22	0.5417	0.4583
scaf1	3248	CHROM:3248:C:T	C	T	C	T	12A1-SC1	30	38	0.4412	0.5588
scaf1	3394	CHROM:3394:T:G	T	G	T	G	12A1-SC1	37	39	0.4868	0.5132
scaf1	3494	CHROM:3494:T:C	T	C	T	C	12A1-SC1	40	36	0.5263	0.4737
scaf1	3636	CHROM:3636:C:G	C	G	C	G	12A1-SC1	34	43	0.4416	0.5584
scaf1	3710	CHROM:3710:C:T	C	T	C	T	12A1-SC1	31	50	0.3827	0.6173
scaf1	3852	CHROM:3852:A:G	A	G	A	G	12A1-SC1	39	32	0.5493	0.4507
scaf1	4008	CHROM:4008:A:G	A	G	A	G	12A1-SC1	41	37	0.5256	0.4744
scaf1	4396	CHROM:4396:T:C	T	C	T	C	12A1-SC1	41	32	0.5616	0.4384
